# Build author features

This notebook is part of the public reproducibility workflow for the study.

Public workflow stage: 05  
Purpose: Construct author-role, affiliation-country, and name-inferred-gender features.

Source notebooks:
- `author_master.ipynb`: stable author UID and role lineage
- `author_msg_check.ipynb`: final countryfill v3 method
- `get_author_gender.ipynb`: Genderize v1 acquisition and unknown-patch lineage
- `get_NCauthor_msg.ipynb`: OpenAlex authorship parsing interface
- `临时数据处理.ipynb`: country-source attribution
- `主表及分析表构建.ipynb`: author enrichment (cell 3)

Default input: future synthetic `sample_author_metadata.csv`, `mock_author_responses.jsonl`, and `sample_papers.csv`.  
Explicit external input: authorized `paper_authors.parquet`, countryfilled v3 with-source table, and Genderize v1 patched table.  
Outputs: `data/work/author_master_enriched.parquet`, `data/work/paper_authors.parquet`, and `data/work/countryfilled_with_source.parquet`.

No OpenAlex or Genderize request is issued by this notebook. External mode reads pre-acquired authorized inputs only. Person-level work outputs must not be published.


In [ ]:
from pathlib import Path
import sys

_places = (Path.cwd().resolve(), Path.cwd().resolve().parent)
_root = next((p for p in _places if (p / "src" / "project_paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter from the public repository root or notebooks directory")
sys.path.insert(0, str(_root))
from src.project_paths import CONFIG_DIR, project_path

try:
    import yaml
except ImportError as exc:
    raise ImportError("PyYAML is required to read the example configuration") from exc
with (CONFIG_DIR / "config.example.yaml").open(encoding="utf-8") as _stream:
    config = yaml.safe_load(_stream)
sample_dir = project_path(config["paths"]["sample_data"])
work_dir = project_path(config["paths"]["work_data"])

import json
import re
import unicodedata
import numpy as np
import pandas as pd

settings = config["author_features"]
mode = settings["mode"]
if mode not in {"mock", "external"}:
    raise ValueError("author_features.mode must be 'mock' or 'external'")
paper_sample_path = sample_dir / "sample_papers.csv"
if mode == "mock" and not paper_sample_path.is_file():
    raise FileNotFoundError(f"Paper sample is not available yet: {paper_sample_path}")
work_dir.mkdir(parents=True, exist_ok=True)
CONFIDENT_PROBABILITY = float(config["genderize"]["confident_probability_threshold"])
CONFIDENT_COUNT = int(config["genderize"]["confident_count_threshold"])
QUERY_FALLBACK_PROBABILITY = float(config["genderize_query"]["fallback_probability_threshold"])
QUERY_FALLBACK_COUNT = int(config["genderize_query"]["fallback_count_threshold"])
assert (CONFIDENT_PROBABILITY, CONFIDENT_COUNT) == (0.80, 20), "Formal confident-gender thresholds must remain 0.80 and 20"


## Keep Genderize acquisition and analysis thresholds distinct

The v1 query/name fallback may consider `probability >= 0.60` and `count >= 10` while choosing a query candidate. That does **not** define confident gender in the analysis. A valid inferred result is `confident` only when `gender_probability >= 0.80` **and** `gender_count >= 20`; otherwise it is `uncertain`. Missing or unusable results are `unknown`. This is a name-based inference, not a person's self-identified gender. No v2 Genderize branch is read.


In [ ]:
def normalize_author_id(value):
    if pd.isna(value):
        return ""
    match = re.search(r"(A\d+)", str(value).strip())
    return match.group(1) if match else ""

def normalize_orcid(value):
    if pd.isna(value):
        return ""
    match = re.search(r"(\d{4}-\d{4}-\d{4}-[\dXx]{4})", str(value).strip())
    return match.group(1).upper() if match else ""

def fold_name(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(character for character in text if not unicodedata.combining(character))
    text = re.sub(r"[\u2010-\u2015–—−]", "-", text.lower().strip())
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text)).strip()

def as_bool(value):
    if pd.isna(value):
        return False
    return value if isinstance(value, bool) else str(value).strip().lower() in {"true", "1", "yes", "y", "t"}

def make_gender_status(gender, probability, count):
    if pd.isna(gender) or str(gender).strip().lower() in {"", "unknown", "none", "nan"}:
        return "unknown"
    probability = pd.to_numeric(probability, errors="coerce")
    count = pd.to_numeric(count, errors="coerce")
    return "confident" if pd.notna(probability) and pd.notna(count) and probability >= CONFIDENT_PROBABILITY and count >= CONFIDENT_COUNT else "uncertain"

def first_nonempty(values):
    for value in values:
        if pd.notna(value) and str(value).strip():
            return str(value).strip()
    return ""

def join_unique(values):
    return " | ".join(sorted(set(
        str(value).strip() for value in values if pd.notna(value) and str(value).strip())))

def choose_country_source(raw_country, filled_country, institution_id="", ror="", affiliation=""):
    if str(raw_country).strip():
        return "original"
    if not str(filled_country).strip():
        return "unknown"
    if str(institution_id).strip():
        return "openalex_inst"
    if str(ror).strip():
        return "ror_org"
    if str(affiliation).strip():
        return "ror_affiliation_chosen"
    return "filled_unknown"


## Read the declared source branch

Mock mode uses synthetic author rows and synthetic response payloads. External mode requires the formal `paper_authors.parquet` input and pre-existing v1/countryfill results under `data/external/`; it never substitutes full or fixed author tables.


In [ ]:
if mode == "mock":
    metadata_path = project_path(settings["sample_author_metadata"])
    responses_path = project_path(settings["mock_responses"])
    for path in (metadata_path, responses_path):
        if not path.is_file():
            raise FileNotFoundError(f"Required synthetic input is not available yet: {path}")
    paper_authors_raw = pd.read_csv(metadata_path)
    response_rows = []
    with responses_path.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                response_rows.append(json.loads(line))
    gender_source = paper_authors_raw.copy()
    country_source = paper_authors_raw.copy()
else:
    external_paths = {name: project_path(settings[key]) for name, key in (
        ("paper_authors", "external_paper_authors"),
        ("countryfilled", "external_countryfilled"),
        ("gender_v1", "external_genderize_v1_patched"))}
    for path in external_paths.values():
        if not path.is_file():
            raise FileNotFoundError(f"Authorized external input is missing: {path}")
    if external_paths["paper_authors"].name != "paper_authors.parquet":
        raise ValueError("Formal author input must be paper_authors.parquet")
    paper_authors_raw = pd.read_parquet(external_paths["paper_authors"])
    country_source = pd.read_parquet(external_paths["countryfilled"])
    gender_source = pd.read_parquet(external_paths["gender_v1"])
    response_rows = []

required = {"paper_code", "author_order"}
if missing := required.difference(paper_authors_raw.columns):
    raise KeyError(f"Formal author input is missing columns: {sorted(missing)}")
paper_authors_raw["paper_code"] = paper_authors_raw["paper_code"].astype(str).str.strip()
paper_authors_raw["author_order"] = pd.to_numeric(paper_authors_raw["author_order"], errors="coerce").astype("Int64")
if paper_authors_raw[["paper_code", "author_order"]].isna().any().any():
    raise ValueError("Author rows require paper and author-order keys")
for role in ("first", "last", "corresponding"):
    column = f"is_{role}_author"
    if column not in paper_authors_raw:
        paper_authors_raw[column] = False
    paper_authors_raw[column] = paper_authors_raw[column].map(as_bool)


## Apply countryfill v3 source attribution and the v1 gender patch

Affiliation-country records are aligned on `(paper_code, author_order)`. For authorized inputs, v3 country values and source labels take precedence; mock payloads only fill synthetic missing values. Gender matching follows the formal priority: author UID within the v1 patch, then OpenAlex author ID, ORCID, and folded name when needed.


In [ ]:
country_columns = [column for column in (
    "paper_code", "author_order", "author_uid", "institution_country_code", "countries",
    "author_affil_country_primary", "institution_country_source", "institution_openalex_id",
    "institution_ror", "raw_affiliation_strings") if column in country_source.columns]
country = country_source[country_columns].copy()
if "author_affil_country_primary" not in country:
    raw = country["institution_country_code"] if "institution_country_code" in country else pd.Series("", index=country.index)
    other = country["countries"] if "countries" in country else pd.Series("", index=country.index)
    country["author_affil_country_primary"] = raw.fillna("").astype(str).where(raw.fillna("").astype(str).str.len().gt(0), other.fillna("").astype(str)).str.split("|").str[0].str.strip()
country["_raw_country"] = country["author_affil_country_primary"].fillna("").astype(str) if mode == "mock" else ""
if mode == "mock":
    for row in response_rows:
        if row.get("provider") not in {"openalex", "ror"}:
            continue
        payload = row.get("payload") or {}
        uid = str(payload.get("author_uid", ""))
        if not uid or "author_uid" not in country:
            continue
        mask = country["author_uid"].astype(str).eq(uid)
        missing = country["author_affil_country_primary"].fillna("").astype(str).str.strip().eq("")
        eligible = mask & missing
        country.loc[eligible, "author_affil_country_primary"] = payload.get("institution_country_code", "")
        country.loc[eligible, "institution_country_source"] = "openalex_inst" if row["provider"] == "openalex" else "ror_org"
if "institution_country_source" not in country:
    country["institution_country_source"] = ""
for index, row in country.iterrows():
    source_value = row.get("institution_country_source", "")
    if pd.isna(source_value) or not str(source_value).strip():
        raw_country = row.get("_raw_country", "") if mode == "mock" else row.get("institution_country_code", "")
        filled = row.get("author_affil_country_primary", "")
        country.at[index, "institution_country_source"] = choose_country_source(
            "" if pd.isna(raw_country) else raw_country,
            "" if pd.isna(filled) else filled,
            row.get("institution_openalex_id", ""), row.get("institution_ror", ""),
            row.get("raw_affiliation_strings", ""))
country_public = country.groupby(["paper_code", "author_order"], as_index=False).agg(
    author_affil_country_primary=("author_affil_country_primary", join_unique),
    institution_country_source=("institution_country_source", join_unique))
country_public["author_affil_country_primary"] = country_public["author_affil_country_primary"].str.split("|").str[0].str.strip()
country_public.to_parquet(work_dir / "countryfilled_with_source.parquet", index=False)

gender = gender_source.copy()
if mode == "mock":
    for row in response_rows:
        if row.get("provider") != "genderize":
            continue
        payload = row.get("payload") or {}
        uid = str(payload.get("author_uid", ""))
        if not uid or "author_uid" not in gender:
            continue
        mask = gender["author_uid"].astype(str).eq(uid)
        for column, key in (("name_inferred_gender", "gender"),
                            ("gender_probability", "probability"), ("gender_count", "count")):
            if column not in gender:
                gender[column] = pd.NA
            missing = gender[column].isna() | gender[column].astype(str).str.strip().eq("")
            gender.loc[mask & missing, column] = payload.get(key)

gender_fields = [column for column in (
    "author_uid", "name_inferred_gender", "gender_probability", "gender_count",
    "genderize_scope", "genderize_fallback", "name_pick_method", "country_id_used") if column in gender]
for column in ("name_inferred_gender", "gender_probability", "gender_count"):
    if column not in gender:
        gender[column] = pd.NA
        gender_fields.append(column)
gender["gender_probability"] = pd.to_numeric(gender["gender_probability"], errors="coerce")
gender["gender_count"] = pd.to_numeric(gender["gender_count"], errors="coerce")
gender["gender_status"] = gender.apply(lambda row: make_gender_status(
    row["name_inferred_gender"], row["gender_probability"], row["gender_count"]), axis=1)

author_features = paper_authors_raw.copy()
if "author_uid" in author_features and "author_uid" in gender:
    lookup = gender[gender_fields + ["gender_status"]].sort_values(
        ["gender_probability", "gender_count"], ascending=False).drop_duplicates("author_uid")
    existing = set(author_features.columns) & set(gender_fields + ["gender_status"])
    existing.discard("author_uid")
    author_features = author_features.drop(columns=sorted(existing))
    author_features = author_features.merge(lookup, on="author_uid", how="left", validate="many_to_one")
else:
    for frame, id_col, orcid_col, name_col in (
        (author_features, "author_openalex_id", "author_orcid", "raw_author_name"),
        (gender, "openalex_author_id", "orcid", "author_name")):
        frame["_aid"] = frame[id_col].map(normalize_author_id) if id_col in frame else ""
        frame["_orcid"] = frame[orcid_col].map(normalize_orcid) if orcid_col in frame else ""
        frame["_name"] = frame[name_col].map(fold_name) if name_col in frame else ""
    author_features["_row"] = np.arange(len(author_features))
    for key, method in (("_aid", "openalex_author_id"), ("_orcid", "orcid"), ("_name", "name_fold")):
        if "name_inferred_gender" in author_features and author_features["name_inferred_gender"].notna().all():
            break
        lookup = gender[gender[key].astype(str).ne("")].sort_values(
            ["gender_probability", "gender_count"], ascending=False).drop_duplicates(key)
        target = author_features[author_features[key].astype(str).ne("")].copy()
        if "name_inferred_gender" in target:
            target = target[target["name_inferred_gender"].isna()]
        matched = target[["_row", key]].merge(
            lookup[[key] + gender_fields + ["gender_status"]], on=key, how="left")
        for field in gender_fields + ["gender_status"]:
            if field in matched:
                if field not in author_features:
                    author_features[field] = pd.NA
                values = matched.set_index("_row")[field]
                author_features.loc[author_features["_row"].isin(values.index), field] = author_features.loc[
                    author_features["_row"].isin(values.index), "_row"].map(values).to_numpy()
        if "gender_match_method" not in author_features:
            author_features["gender_match_method"] = ""
        author_features.loc[author_features["_row"].isin(matched.loc[matched["name_inferred_gender"].notna(), "_row"]), "gender_match_method"] = method
    author_features = author_features.drop(columns=["_aid", "_orcid", "_name", "_row"])

if "gender_status" not in author_features:
    author_features["gender_status"] = "unknown"
author_features["gender_status"] = author_features.apply(lambda row: make_gender_status(
    row.get("name_inferred_gender"), row.get("gender_probability"), row.get("gender_count")), axis=1)
country_for_join = country_public.rename(columns={"author_affil_country_primary": "_country_v3"})
author_features = author_features.merge(country_for_join, on=["paper_code", "author_order"], how="outer", suffixes=("", "_country"))
base_country = author_features["author_affil_country_primary"] if "author_affil_country_primary" in author_features else pd.Series("", index=author_features.index)
author_features["author_affil_country_primary"] = author_features["_country_v3"].fillna(base_country).fillna("").astype(str).replace({"nan": "", "None": ""}).str.split("|").str[0].str.strip()
author_features = author_features.drop(columns="_country_v3")
if "institution_country_source_country" in author_features:
    base_source = author_features["institution_country_source"] if "institution_country_source" in author_features else pd.Series("", index=author_features.index)
    author_features["institution_country_source"] = author_features["institution_country_source_country"].fillna(base_source)
    author_features = author_features.drop(columns="institution_country_source_country")
author_features["gender_status"] = author_features["gender_status"].fillna("unknown")


## Write minimal work tables

Only join keys, role flags, country source, and inferred-gender fields needed downstream are written. Raw names, ORCIDs, OpenAlex author IDs, affiliations, DOI, and response payloads remain in memory and are not copied to public work tables.


In [ ]:
paper_columns = [column for column in (
    "paper_code", "author_order", "author_uid", "is_first_author", "is_last_author",
    "is_corresponding_author", "author_affil_country_primary", "institution_country_source",
    "name_inferred_gender", "gender_probability", "gender_count", "gender_status") if column in author_features]
paper_authors = author_features[paper_columns].copy()
paper_authors.to_parquet(work_dir / "paper_authors.parquet", index=False)
master_columns = [column for column in (
    "author_uid", "name_inferred_gender", "gender_probability", "gender_count",
    "gender_status", "genderize_scope", "genderize_fallback", "name_pick_method",
    "country_id_used") if column in author_features]
author_master_enriched = author_features[master_columns].drop_duplicates("author_uid") if "author_uid" in master_columns else author_features[master_columns].drop_duplicates()
author_master_enriched.to_parquet(work_dir / "author_master_enriched.parquet", index=False)
print(f"Saved {len(paper_authors)} paper-author rows and {len(author_master_enriched)} author-feature rows")
